In [1]:
print(123)

123


In [10]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [11]:
from starter import rag
from rag_traced import RAGTraced

In [12]:
traced_rag = RAGTraced(
    index=rag.index, 
    llm_client=rag.llm_client,  
    instructions=rag.instructions,
    prompt_template=rag.prompt_template,
    model=rag.model
)


In [41]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = traced_rag.rag(query)
print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0xf1d5b11381b898061767b782715de91c",
        "span_id": "0x9060967d96f77f25",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x98d6289d585d7a7c",
    "start_time": "2026-07-16T17:30:51.993479Z",
    "end_time": "2026-07-16T17:30:51.999467Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?",
        "num_results": 5,
        "actual_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "67e271c3-b1e1-4291-bff7-3a1aaf5e9ae6",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "tr

 **Q1:** How many spans does the trace produce?  
 **Answer:** 3

In [14]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = traced_rag.rag(query)

print("Answer:", answer)
print("\n--- Token Usage ---")
print(f"Input tokens: {traced_rag.last_tokens.get('input_tokens', 0)}")
print(f"Output tokens: {traced_rag.last_tokens.get('output_tokens', 0)}")
print(f"Total tokens: {traced_rag.last_tokens.get('total_tokens', 0)}")
print(f"Cost: ${traced_rag.last_tokens.get('total_cost_usd', 0):.6f}")

{
    "name": "search",
    "context": {
        "trace_id": "0x95f51bf2d9a9a1dec4702d5debb00259",
        "span_id": "0x8dcf22a338725b38",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xa6ca2c3a72a6f36a",
    "start_time": "2026-07-16T17:24:13.657879Z",
    "end_time": "2026-07-16T17:24:13.660023Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?",
        "num_results": 5,
        "actual_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "67e271c3-b1e1-4291-bff7-3a1aaf5e9ae6",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "tr

**Q2:** How many input tokens do we see?  
**Answer:** 7000

In [15]:
from datetime import datetime

start = datetime.fromisoformat("2026-07-16T16:51:40.784252Z".replace('Z', '+00:00'))
end = datetime.fromisoformat("2026-07-16T16:51:42.043347Z".replace('Z', '+00:00'))
duration_ms = (end - start).total_seconds() * 1000
print(f"Duration: {duration_ms:.0f}ms")


Duration: 1259ms


**Q3:** For a typical query, roughly how long does the LLM call take?  
**Answer:** 500-2000ms

In [42]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [43]:
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)

In [44]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = traced_rag.rag(query)
print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0x847767c9c19e9c226723fea325387bc4",
        "span_id": "0xe353c28180a964e7",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x8b1259509697c72b",
    "start_time": "2026-07-16T17:31:22.785283Z",
    "end_time": "2026-07-16T17:31:22.788523Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?",
        "num_results": 5,
        "actual_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "67e271c3-b1e1-4291-bff7-3a1aaf5e9ae6",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "tr

**Q4:** Which span names appear in the spans table?  
**Answer:** rag, search, and llm

**Q5:** Which span type takes the most total time?  
**Answer:** llm

In [46]:
query = "How does the agentic loop keep calling the model until it stops?"

for i in range(4):
    print(f"\n--- Run {i+1} ---")
    answer = traced_rag.rag(query)
    print(f"Input tokens: {traced_rag.last_tokens.get('input_tokens', 0)}")
    print(f"Output tokens: {traced_rag.last_tokens.get('output_tokens', 0)}")


trace.get_tracer_provider().force_flush()


--- Run 1 ---
{
    "name": "search",
    "context": {
        "trace_id": "0x2b2ff461eac38ed3338522ab43a46d3e",
        "span_id": "0x81afadb15d776524",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xb48f99f4bb588556",
    "start_time": "2026-07-16T17:34:50.589475Z",
    "end_time": "2026-07-16T17:34:50.606779Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?",
        "num_results": 5,
        "actual_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "67e271c3-b1e1-4291-bff7-3a1aaf5e9ae6",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context"

True

**Q6:** How much do the input tokens vary across these 4 runs?  
**Answer:** They're identical